<a href="https://colab.research.google.com/github/antonum/Redis-Workshops/blob/experimental/Feast_Redis_CreditScoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redis as online feature store with Feast - setup

This notebook is an adaptation of the [Feast Quickstart](https://docs.feast.dev/getting-started/quickstart) that uses [Redis online feature store](https://docs.feast.dev/reference/online-stores/redis), instead of default SQLite.

In [1]:
!pip install -q feast['redis']
!feast version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.5/166.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.1/241.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.2/243.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.1/442.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/7

In [2]:
%%bash
mkdir creditscore
mkdir creditscore/data
wget https://github.com/antonum/feast-redis/raw/refs/heads/main/creditscore/data/credit_history.parquet -q -P creditscore/data
wget https://github.com/antonum/feast-redis/raw/refs/heads/main/creditscore/data/zipcode_table.parquet -q -P creditscore/data
wget https://github.com/antonum/feast-redis/raw/refs/heads/main/creditscore/data/loan_table.parquet -q -P creditscore/data
#cd creditscore
#touch __init__.py


In [4]:
REDIS_URL_FEAST="localhost:6379,ssl=false,password="
feature_store = \
f"""project: creditscore
registry: data/registry.db
provider: local
online_store:
    #path: data/online_store.db
    type: redis
    connection_string: {REDIS_URL_FEAST}
entity_key_serialization_version: 2
"""
with open('creditscore/feature_store.yaml', "w") as file:
    file.write(feature_store)

# Print our feature_store.yaml
! cat creditscore/feature_store.yaml

project: creditscore
registry: data/registry.db
provider: local
online_store:
    #path: data/online_store.db
    type: redis
    connection_string: localhost:6379,ssl=false,password=
entity_key_serialization_version: 2


In [5]:
features_file = \
f"""from datetime import timedelta

from feast import (Entity, Field, FeatureView,
                   ValueType, FileSource)

from feast.types import Float32, Int64, String

zipcode = Entity(
    name="zipcode"
    )

zipcode_source = FileSource(
    path="data/zipcode_table.parquet",
    timestamp_field="event_timestamp",
    #event_timestamp_column="event_timestamp",
    created_timestamp_column="created_timestamp",
)

zipcode_features = FeatureView(
    name="zipcode_features",
    entities=[zipcode],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="city", dtype=String),
        Field(name="state", dtype=String),
        Field(name="location_type", dtype=String),
        Field(name="tax_returns_filed", dtype=Int64),
        Field(name="population", dtype=Int64),
        Field(name="total_wages", dtype=Int64),
    ],
    source=zipcode_source,
)

dob_ssn = Entity(
    name="dob_ssn",
    description="Date of birth and last four digits of social security number",
)

credit_history_source = FileSource(
    path="data/credit_history.parquet",
    timestamp_field="event_timestamp",
    #event_timestamp_column="event_timestamp",
    created_timestamp_column="created_timestamp",

)

credit_history = FeatureView(
    name="credit_history",
    entities=[dob_ssn],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="dob_ssn", dtype=String),  # Add entity column for dob_ssn
        Field(name="credit_card_due", dtype=Int64),
        Field(name="mortgage_due", dtype=Int64),
        Field(name="student_loan_due", dtype=Int64),
        Field(name="vehicle_loan_due", dtype=Int64),
        Field(name="hard_pulls", dtype=Int64),
        Field(name="missed_payments_2y", dtype=Int64),
        Field(name="missed_payments_1y", dtype=Int64),
        Field(name="missed_payments_6m", dtype=Int64),
        Field(name="bankruptcies", dtype=Int64),
    ],
    source=credit_history_source,
)
"""
with open('creditscore/features.py', "w") as file:
    file.write(features_file)

# Print our features.py
#! cat creditscore/features.py

In [6]:
%cd creditscore/
!feast apply

/content/creditscore
No project found in the repository. Using project name creditscore defined in feature_store.yaml
Applying changes for project creditscore
Deploying infrastructure for zipcode_features
Deploying infrastructure for credit_history


In [7]:
%%sh
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install redis-stack-server  > /dev/null 2>&1
redis-stack-server --daemonize yes

deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb jammy main
Starting redis-stack-server, database path /var/lib/redis-stack


In [8]:
import os

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", "6379")
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
# Replace values above with your own if using Redis Cloud instance
#REDIS_HOST="redis-18374.c253.us-central1-1.gce.cloud.redislabs.com"
#REDIS_PORT=18374
#REDIS_PASSWORD="1TNxTEdYRDgIDKM2gDfasupCADXXXX"

# Shortcut for redis-cli $REDIS_CONN command
# If SSL is enabled on the endpoint add --tls
if REDIS_PASSWORD!="":
  os.environ["REDIS_CONN"]=f"-h {REDIS_HOST} -p {REDIS_PORT} -a {REDIS_PASSWORD} --no-auth-warning"
else:
  os.environ["REDIS_CONN"]=f"-h {REDIS_HOST} -p {REDIS_PORT}"

# If SSL is enabled on the endpoint, use rediss:// as the URL prefix
REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

# See https://docs.feast.dev/reference/online-stores/redis for details on Feast connection to Redis
REDIS_URL_FEAST = f"{REDIS_HOST}:{REDIS_PORT},ssl=false,password={REDIS_PASSWORD}"

In [9]:
!feast materialize 2010-11-19T16:57:10 2024-11-26T16:57:10



Materializing 2 feature views from 2010-11-19 16:57:10+00:00 to 2024-11-26 16:57:10+00:00 into the redis online store.

zipcode_features:
100%|███████████████████████████████████████████████████████| 28844/28844 [00:02<00:00, 9803.30it/s]
credit_history:
100%|███████████████████████████████████████████████████████| 28633/28633 [00:03<00:00, 9185.67it/s]


In [ ]:
%cd my_project/feature_repo
%ls

/content/my_project/feature_repo
data/  example_repo.py  feature_store.yaml  __init__.py  __pycache__/  test_workflow.py


In [ ]:
#!pwd
!cat feature_store.yaml

project: my_project
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 2
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth


In [ ]:
feature_store = \
f"""project: my_project
registry: data/registry.db
provider: local
online_store:
    #path: data/online_store.db
    type: redis
    connection_string: {REDIS_URL_FEAST}
"""
with open('feature_store.yaml', "w") as feature_store_file:
    feature_store_file.write(feature_store)

# Print our feature_store.yaml
! cat feature_store.yaml

project: my_project
registry: data/registry.db
provider: local
online_store:
    #path: data/online_store.db
    type: redis
    connection_string: localhost:6379,ssl=false,password=


In [ ]:
import pandas as pd
pd.read_parquet("data/driver_stats.parquet")

,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
0,2024-11-10 15:00:00+00:00,1005,0.683643,0.123935,717,2024-11-25 15:33:32.117
1,2024-11-10 16:00:00+00:00,1005,0.163229,0.934807,685,2024-11-25 15:33:32.117
2,2024-11-10 17:00:00+00:00,1005,0.387899,0.754735,774,2024-11-25 15:33:32.117
3,2024-11-10 18:00:00+00:00,1005,0.862102,0.814070,91,2024-11-25 15:33:32.117
4,2024-11-10 19:00:00+00:00,1005,0.484867,0.121068,678,2024-11-25 15:33:32.117
...,...,...,...,...,...,...
1802,2024-11-25 13:00:00+00:00,1001,0.416769,0.924485,407,2024-11-25 15:33:32.117
1803,2024-11-25 14:00:00+00:00,1001,0.813480,0.211901,941,2024-11-25 15:33:32.117
1804,2021-04-12 07:00:00+00:00,1001,0.917656,0.985833,919,2024-11-25 15:33:32.117
1805,2024-11-18 03:00:00+00:00,1003,0.487243,0.179945,627,2024-11-25 15:33:32.117


In [ ]:
!feast apply

/usr/local/lib/python3.10/dist-packages/feast/repo_config.py:250: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(
Applying changes for project my_project
/usr/local/lib/python3.10/dist-packages/feast/feature_store.py:580: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
Deploying infrastructure for driver_hourly_stats_fresh
Deploying infrastructure for driver_hourly_stats


In [ ]:
from datetime import datetime
import pandas as pd

from feast import FeatureStore

# Note: see https://docs.feast.dev/getting-started/concepts/feature-retrieval for
# more details on how to retrieve for all entities in the offline store instead
entity_df = pd.DataFrame.from_dict(
    {
        # entity's join key -> entity values
        "driver_id": [1001, 1002, 1003],
        # "event_timestamp" (reserved key) -> timestamps
        "event_timestamp": [
            datetime(2021, 4, 12, 10, 59, 42),
            datetime(2021, 4, 12, 8, 12, 10),
            datetime(2021, 4, 12, 16, 40, 26),
        ],
        # (optional) label name -> label values. Feast does not process these
        "label_driver_reported_satisfaction": [1, 5, 3],
        # values we're using for an on-demand transformation
        "val_to_add": [1, 2, 3],
        "val_to_add_2": [10, 20, 30],
    }
)

store = FeatureStore(repo_path=".")

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
        "transformed_conv_rate:conv_rate_plus_val1",
        "transformed_conv_rate:conv_rate_plus_val2",
    ],
).to_df()

print("----- Feature schema -----\n")
print(training_df.info())

print()
print("----- Example features -----\n")
print(training_df.head())

----- Feature schema -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   driver_id                           3 non-null      int64              
 1   event_timestamp                     3 non-null      datetime64[ns, UTC]
 2   label_driver_reported_satisfaction  3 non-null      int64              
 3   val_to_add                          3 non-null      int64              
 4   val_to_add_2                        3 non-null      int64              
 5   conv_rate                           3 non-null      float32            
 6   acc_rate                            3 non-null      float32            
 7   avg_daily_trips                     3 non-null      int32              
 8   conv_rate_plus_val1                 3 non-null      float64            
 9   conv_rate_plus_val2

/usr/local/lib/python3.10/dist-packages/feast/repo_config.py:250: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(


In [ ]:
entity_df["event_timestamp"] = pd.to_datetime("now", utc=True)
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
        "transformed_conv_rate:conv_rate_plus_val1",
        "transformed_conv_rate:conv_rate_plus_val2",
    ],
).to_df()

print("\n----- Example features -----\n")
print(training_df.head())

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)



----- Example features -----

   driver_id                  event_timestamp  \
0       1001 2024-11-25 15:33:48.030176+00:00   
1       1002 2024-11-25 15:33:48.030176+00:00   
2       1003 2024-11-25 15:33:48.030176+00:00   

   label_driver_reported_satisfaction  val_to_add  val_to_add_2  conv_rate  \
0                                   1           1            10   0.813480   
1                                   5           2            20   0.013196   
2                                   3           3            30   0.648087   

   acc_rate  avg_daily_trips  conv_rate_plus_val1  conv_rate_plus_val2  
0  0.211901              941             1.813480            10.813480  
1  0.139228              255             2.013196            20.013196  
2  0.213400              928             3.648087            30.648087  


In [ ]:
%%bash
CURRENT_TIME=$(date -u +"%Y-%m-%dT%H:%M:%S")
# For mac
LAST_YEAR=$(date -u -v -1y +"%Y-%m-%dT%H:%M:%S")
# For Linux
# LAST_YEAR=$(date -u -d "last year" +"%Y-%m-%dT%H:%M:%S")

feast materialize-incremental $LAST_YEAR $CURRENT_TIME

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Materializing 2 feature views to 2024-11-25 15:33:48+00:00 into the redis online store.

driver_hourly_stats_fresh from 2024-11-24 15:33:52+00:00 to 2024-11-25 15:33:48+00:00:
driver_hourly_stats from 2024-11-24 15:33:52+00:00 to 2024-11-25 15:33:48+00:00:


date: invalid option -- 'v'
Try 'date --help' for more information.
/usr/local/lib/python3.10/dist-packages/feast/repo_config.py:250: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(
100%|███████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 2597.74it/s]


In [ ]:
from pprint import pprint
from feast import FeatureStore

store = FeatureStore(repo_path=".")

feature_vector = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    entity_rows=[
        # {join_key: entity_value}
        {"driver_id": 1004},
        {"driver_id": 1005},
    ],
).to_dict()

pprint(feature_vector)

{'acc_rate': [0.5636054873466492, 0.9438936114311218],
 'avg_daily_trips': [62, 890],
 'conv_rate': [0.5937009453773499, 0.9745711088180542],
 'driver_id': [1004, 1005]}


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/feast/repo_config.py:250: RuntimeWarning: `entity_key_serialization_version` is either not specified in the feature_store.yaml, or is specified to a value <= 1.This serialization version may cause errors when trying to write fields with the `Long` data type into the online store. Specifying `entity_key_serialization_version` to 2 is recommended for new projects. 
  warnings.warn(


In [ ]:
!redis-cli keys "*"

1) "\x02\x00\x00\x00driver_id\x04\x00\x00\x00\x04\x00\x00\x00\xec\x03\x00\x00my_project"
2) "\x02\x00\x00\x00driver_id\x04\x00\x00\x00\x04\x00\x00\x00\xe9\x03\x00\x00my_project"
3) "\x02\x00\x00\x00driver_id\x04\x00\x00\x00\x04\x00\x00\x00\xea\x03\x00\x00my_project"
4) "\x02\x00\x00\x00driver_id\x04\x00\x00\x00\x04\x00\x00\x00\xed\x03\x00\x00my_project"
5) "\x02\x00\x00\x00driver_id\x04\x00\x00\x00\x04\x00\x00\x00\xeb\x03\x00\x00my_project"
